In [ ]:
import os
import requests
import pandas as pd

# ── User: Set this to wherever you want your "Studies_Workbench" folder to live ──
output_base_path = '/path/to/your/desired/folder'  
# ────────────────────────────────────────────────────────────────────────────────

# This will create: {output_base_path}/Studies_Workbench/{STUDY_ID}/{ANALYSIS_ID}.csv
base_out = os.path.join(output_base_path, 'Studies_Workbench')
os.makedirs(base_out, exist_ok=True)

# List your studies here
studies = [
    'ST000041', 'ST002829', 'ST001420', 'ST001789',
    'ST001412', 'ST002016', 'ST001736', 'ST002301',
    'ST001933', 'ST000899', 'ST000284', 'ST002428',
    'ST000974', 'ST002100', 'ST001940', 'ST002498'
]

BASE_API = 'https://www.metabolomicsworkbench.org/rest'

for study in studies:
    print(f'\n▶ Processing study {study}')
    # fetch analysis IDs
    resp = requests.get(f'{BASE_API}/study/{study}/analysis').json()
    if not resp:
        print('  ⚠️  No assays found')
        continue

    if list(resp.keys())[0] != '1':
        analysis_ids = [resp['analysis_id']]
    else:
        df = pd.DataFrame.from_dict(resp, orient='index')
        analysis_ids = df['analysis_id'].tolist()
    print(f'  Found {len(analysis_ids)} assay(s)')

    study_dir = os.path.join(base_out, study)
    os.makedirs(study_dir, exist_ok=True)

    for aid in analysis_ids:
        url = f'{BASE_API}/study/{aid}/datatable/file'
        df = pd.read_csv(url, sep='\t')
        out_path = os.path.join(study_dir, f'{aid}.csv')
        df.to_csv(out_path, index=False)
        print(f'    • Saved {out_path}')
